# Assign acoustic-unit tokens

In [ ]:
from __future__ import annotations

import json
import logging
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List

import joblib
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

In [ ]:
# --- Project root + config setup ---
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "configs" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. "
        "Expected to find configs/config.yaml in the project root."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("CONFIG_PATH  =", CONFIG_PATH)

DEFAULT_CONFIG = {
    "paths": {
        "output_dir": "output",
        "log_dir": "output/logs",
    },
    "segmentation": {
        "output_subdir": "02_segments",
    },
    "clustering": {
        "segment_vectors_subdir": "03_segment_vectors",
        "kstudy_subdir": "04_kstudy",
    },
    "tokens": {
        "output_subdir": "05_tokens",
        "overwrite": False,
        "token_prefix": "u",
        "token_zero_pad": None,
        "round_time_decimals": 4,
        "include_manifest_metadata_in_jsonl": True,
    },
    "manifest": {
        "output_subdir": "00_manifest",
    },
}

def deep_update(base: Dict[str, Any], updates: Dict[str, Any]) -> Dict[str, Any]:
    result = deepcopy(base)
    for key, value in (updates or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update(result[key], value)
        else:
            result[key] = value
    return result

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        user_config = yaml.safe_load(f) or {}
else:
    user_config = {}

config = deep_update(DEFAULT_CONFIG, user_config)

print("\nResolved config paths:")
print("segmentation.output_subdir        =", config["segmentation"]["output_subdir"])
print("clustering.segment_vectors_subdir =", config["clustering"]["segment_vectors_subdir"])
print("clustering.kstudy_subdir          =", config["clustering"]["kstudy_subdir"])
print("tokens.output_subdir              =", config["tokens"]["output_subdir"])
print("manifest.output_subdir            =", config["manifest"]["output_subdir"])

config


In [ ]:
# --- Derived paths + logging ---
OUTPUT_DIR = PROJECT_ROOT / config["paths"]["output_dir"]
LOG_DIR = PROJECT_ROOT / config["paths"].get("log_dir", "output/logs")

SEG_DIR = OUTPUT_DIR / config["segmentation"]["output_subdir"]
SEGMENT_VECTOR_DIR = OUTPUT_DIR / config["clustering"]["segment_vectors_subdir"]
KSTUDY_DIR = OUTPUT_DIR / config["clustering"]["kstudy_subdir"]
TOKEN_DIR = OUTPUT_DIR / config["tokens"]["output_subdir"]
MANIFEST_DIR = OUTPUT_DIR / config["manifest"]["output_subdir"]

for p in [TOKEN_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEGMENTS_CSV = SEG_DIR / "segments.csv"
SEGMENT_VECTORS_NPZ = SEGMENT_VECTOR_DIR / "segment_vectors.npz"
MODEL_PKL = KSTUDY_DIR / "kmeans_model.pkl"
MANIFEST_CSV = MANIFEST_DIR / "manifest.csv"

TOKEN_JSONL = TOKEN_DIR / "token_sequences.jsonl"
TOKEN_COUNTS_CSV = TOKEN_DIR / "token_counts_by_clip.csv"
GLOBAL_FREQ_CSV = TOKEN_DIR / "global_token_frequencies.csv"

overwrite = bool(config["tokens"].get("overwrite", False))

LOG_FILE = LOG_DIR / "04_assign_unit_tokens.log"
logger = logging.getLogger("04_assign_unit_tokens")
logger.setLevel(logging.INFO)
logger.handlers.clear()

fh = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(sh)

print("SEGMENTS_CSV     =", SEGMENTS_CSV)
print("SEGMENT_VECTORS  =", SEGMENT_VECTORS_NPZ)
print("MODEL_PKL        =", MODEL_PKL)
print("MANIFEST_CSV     =", MANIFEST_CSV)
print("TOKEN_JSONL      =", TOKEN_JSONL)
print("TOKEN_COUNTS_CSV =", TOKEN_COUNTS_CSV)
print("GLOBAL_FREQ_CSV  =", GLOBAL_FREQ_CSV)
print("LOG_FILE         =", LOG_FILE)

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Token output dir: %s", TOKEN_DIR)


In [ ]:
# --- Load required inputs ---
if not SEGMENTS_CSV.exists():
    raise FileNotFoundError(f"Missing required file: {SEGMENTS_CSV}")
if not SEGMENT_VECTORS_NPZ.exists():
    raise FileNotFoundError(f"Missing required file: {SEGMENT_VECTORS_NPZ}")
if not MODEL_PKL.exists():
    raise FileNotFoundError(f"Missing required file: {MODEL_PKL}")

segments_df = pd.read_csv(SEGMENTS_CSV)
model_bundle = joblib.load(MODEL_PKL)
sv = np.load(SEGMENT_VECTORS_NPZ, allow_pickle=False)

manifest_df = None
if MANIFEST_CSV.exists():
    manifest_df = pd.read_csv(MANIFEST_CSV)

print("segments_df shape:", segments_df.shape)
print("segment_vectors keys:", sv.files)
print("manifest exists:", manifest_df is not None)

display(segments_df.head())
if manifest_df is not None:
    display(manifest_df.head())

logger.info("Loaded %d segment rows", len(segments_df))
logger.info("Loaded model bundle from %s", MODEL_PKL)

In [ ]:
# --- Integrity checks and reconstruction of the clustering input ---
required_segment_cols = [
    "segment_id", "clip_id", "segment_index", "start_sec", "end_sec", "duration_sec"
]
missing_segment_cols = [c for c in required_segment_cols if c not in segments_df.columns]
if missing_segment_cols:
    raise ValueError(f"segments.csv is missing required columns: {missing_segment_cols}")

required_npz_keys = [
    "vectors", "segment_ids", "clip_ids", "segment_index", "start_sec", "end_sec", "duration_sec"
]
missing_npz_keys = [k for k in required_npz_keys if k not in sv.files]
if missing_npz_keys:
    raise ValueError(f"segment_vectors.npz is missing required arrays: {missing_npz_keys}")

vector_matrix = sv["vectors"].astype(np.float32)
vector_segment_ids = sv["segment_ids"].astype(str)
vector_clip_ids = sv["clip_ids"].astype(str)
vector_segment_index = sv["segment_index"].astype(np.int32)
vector_start_sec = sv["start_sec"].astype(np.float32)
vector_end_sec = sv["end_sec"].astype(np.float32)
vector_duration_sec = sv["duration_sec"].astype(np.float32)

print("vector_matrix shape:", vector_matrix.shape)
print("unique clips in vectors:", len(np.unique(vector_clip_ids)))

if len(vector_segment_ids) != len(segments_df):
    print("Note: segment_vectors row count and segments.csv row count differ; alignment will be based on segment_id.")
    logger.warning("Segment count mismatch: vectors=%d vs segments=%d", len(vector_segment_ids), len(segments_df))

vector_df = pd.DataFrame({
    "segment_id": vector_segment_ids,
    "clip_id_from_vectors": vector_clip_ids,
    "segment_index_from_vectors": vector_segment_index,
    "start_sec_from_vectors": vector_start_sec,
    "end_sec_from_vectors": vector_end_sec,
    "duration_sec_from_vectors": vector_duration_sec,
    "row_ix": np.arange(len(vector_segment_ids), dtype=np.int32),
})

merged_df = segments_df.merge(vector_df, on="segment_id", how="inner", validate="one_to_one")
if len(merged_df) != len(vector_df):
    raise ValueError(
        f"After merge, {len(merged_df)} rows remain but {len(vector_df)} vector rows exist. "
        "This suggests missing/duplicate segment_id values."
    )

# Keep exactly the vector row order for correct label assignment.
merged_df = merged_df.sort_values("row_ix").reset_index(drop=True)

# Reconstruct the clustering input with the saved scaler/PCA from Step 03.
scaler = model_bundle["scaler"]
pca = model_bundle["pca"]
kmeans_model = model_bundle["model"]
selected_k = int(model_bundle["selected_k"])

X_scaled = scaler.transform(vector_matrix)
if pca is not None:
    X_cluster = pca.transform(X_scaled)
else:
    X_cluster = X_scaled

print("selected_k =", selected_k)
print("X_cluster shape:", X_cluster.shape)
print("unique segments after merge:", merged_df["segment_id"].nunique())

logger.info("Selected K from model bundle: %d", selected_k)
logger.info("Clustering input shape after transform: %s", X_cluster.shape)

In [ ]:
# --- Predict token/unit labels for all segments ---
pred_labels = kmeans_model.predict(X_cluster).astype(np.int32)

if pred_labels.shape[0] != merged_df.shape[0]:
    raise ValueError("Predicted labels length does not match merged segment table length.")

zero_pad = config["tokens"]["token_zero_pad"]
if zero_pad is None:
    zero_pad = max(2, len(str(selected_k - 1)))

token_prefix = str(config["tokens"]["token_prefix"])

def make_token_id(unit_index: int) -> str:
    return f"{token_prefix}{int(unit_index):0{zero_pad}d}"

merged_df["unit_index"] = pred_labels
merged_df["unit_id"] = merged_df["unit_index"].map(make_token_id)

# Preserve clip order from manifest if possible; otherwise use first appearance order in segments.csv.
if manifest_df is not None and "clip_id" in manifest_df.columns:
    clip_order_map = {clip_id: i for i, clip_id in enumerate(manifest_df["clip_id"].astype(str).tolist())}
    merged_df["clip_order"] = merged_df["clip_id"].astype(str).map(clip_order_map).fillna(10**9).astype(int)
else:
    first_clip_order = {clip_id: i for i, clip_id in enumerate(merged_df["clip_id"].astype(str).drop_duplicates().tolist())}
    merged_df["clip_order"] = merged_df["clip_id"].astype(str).map(first_clip_order).astype(int)

merged_df = merged_df.sort_values(
    ["clip_order", "clip_id", "segment_index", "start_sec", "end_sec", "segment_id"]
).reset_index(drop=True)

display(
    merged_df[
        ["clip_id", "segment_id", "segment_index", "start_sec", "end_sec", "duration_sec", "unit_index", "unit_id"]
    ].head(20)
)

print("Unique units assigned:", merged_df["unit_index"].nunique())
print("Unit count preview:")
display(merged_df["unit_id"].value_counts().sort_index().head(10))

logger.info("Assigned %d unique units across %d segments", merged_df["unit_index"].nunique(), len(merged_df))

In [ ]:
# --- Build ordered token sequences per clip ---
round_decimals = int(config["tokens"]["round_time_decimals"])
include_manifest_metadata = bool(config["tokens"]["include_manifest_metadata_in_jsonl"])

manifest_meta_map: Dict[str, Dict[str, Any]] = {}
if include_manifest_metadata and manifest_df is not None and "clip_id" in manifest_df.columns:
    manifest_cols_to_skip = {"audio_path", "audio_relpath"}
    manifest_cols = [c for c in manifest_df.columns if c not in manifest_cols_to_skip]
    for row in manifest_df[manifest_cols].itertuples(index=False):
        row_dict = dict(zip(manifest_cols, row))
        manifest_meta_map[str(row_dict["clip_id"])] = row_dict

sequence_records: List[Dict[str, Any]] = []
for clip_id, g in tqdm(merged_df.groupby("clip_id", sort=False), desc="Building token sequences"):
    g = g.sort_values(["segment_index", "start_sec", "end_sec", "segment_id"]).reset_index(drop=True)

    seq_tokens = g["unit_id"].astype(str).tolist()
    seq_units = g["unit_index"].astype(int).tolist()

    segments_payload = []
    for r in g.itertuples(index=False):
        segments_payload.append({
            "segment_id": str(r.segment_id),
            "segment_index": int(r.segment_index),
            "unit_index": int(r.unit_index),
            "unit_id": str(r.unit_id),
            "start_sec": round(float(r.start_sec), round_decimals),
            "end_sec": round(float(r.end_sec), round_decimals),
            "duration_sec": round(float(r.duration_sec), round_decimals),
        })

    record = {
        "clip_id": str(clip_id),
        "n_segments": int(len(g)),
        "sequence_unit_ids": seq_tokens,
        "sequence_unit_indices": seq_units,
        "segments": segments_payload,
    }

    if "clip_duration_sec" in g.columns:
        record["clip_duration_sec"] = round(float(g["clip_duration_sec"].iloc[0]), round_decimals)

    if include_manifest_metadata:
        record["metadata"] = manifest_meta_map.get(str(clip_id), {})

    sequence_records.append(record)

print("Number of token sequences:", len(sequence_records))
display(pd.DataFrame({
    "clip_id": [r["clip_id"] for r in sequence_records[:5]],
    "n_segments": [r["n_segments"] for r in sequence_records[:5]],
    "first_tokens": [", ".join(r["sequence_unit_ids"][:8]) for r in sequence_records[:5]],
}))

logger.info("Built %d token sequences", len(sequence_records))

In [ ]:
# --- Save token_sequences.jsonl ---
if TOKEN_JSONL.exists() and not overwrite:
    print(f"token_sequences.jsonl already exists and overwrite=false -> keeping existing file: {TOKEN_JSONL}")
    logger.info("token_sequences.jsonl exists and overwrite=false; file not rewritten")
else:
    with open(TOKEN_JSONL, "w", encoding="utf-8") as f:
        for rec in sequence_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print("Saved:", TOKEN_JSONL)
    logger.info("Saved token sequence JSONL to %s", TOKEN_JSONL)

TOKEN_JSONL

In [ ]:
# --- Build and save token_counts_by_clip.csv ---
clip_count_rows = []
for clip_id, g in merged_df.groupby("clip_id", sort=False):
    g = g.sort_values(["segment_index", "start_sec", "end_sec", "segment_id"]).reset_index(drop=True)
    n_segments_in_clip = int(len(g))
    counts = g.groupby(["unit_index", "unit_id"], as_index=False).size().rename(columns={"size": "count"})
    first_pos = g.groupby(["unit_index", "unit_id"], as_index=False)["segment_index"].min().rename(columns={"segment_index": "first_segment_index"})
    last_pos = g.groupby(["unit_index", "unit_id"], as_index=False)["segment_index"].max().rename(columns={"segment_index": "last_segment_index"})
    count_df = counts.merge(first_pos, on=["unit_index", "unit_id"]).merge(last_pos, on=["unit_index", "unit_id"])
    count_df["clip_id"] = str(clip_id)
    count_df["n_segments_in_clip"] = n_segments_in_clip
    count_df["proportion_of_clip_segments"] = count_df["count"] / max(n_segments_in_clip, 1)
    clip_count_rows.append(count_df)

token_counts_by_clip_df = pd.concat(clip_count_rows, ignore_index=True)
token_counts_by_clip_df = token_counts_by_clip_df[
    ["clip_id", "unit_index", "unit_id", "count", "proportion_of_clip_segments", "first_segment_index", "last_segment_index", "n_segments_in_clip"]
].sort_values(["clip_id", "unit_index"]).reset_index(drop=True)

if TOKEN_COUNTS_CSV.exists() and not overwrite:
    print(f"token_counts_by_clip.csv already exists and overwrite=false -> keeping existing file: {TOKEN_COUNTS_CSV}")
    logger.info("token_counts_by_clip.csv exists and overwrite=false; file not rewritten")
else:
    token_counts_by_clip_df.to_csv(TOKEN_COUNTS_CSV, index=False)
    print("Saved:", TOKEN_COUNTS_CSV)
    logger.info("Saved token_counts_by_clip.csv to %s", TOKEN_COUNTS_CSV)

display(token_counts_by_clip_df.head(20))
print("Rows in token_counts_by_clip_df:", len(token_counts_by_clip_df))

In [ ]:
# --- Build and save global_token_frequencies.csv ---
n_total_segments = int(len(merged_df))
n_total_clips = int(merged_df["clip_id"].nunique())

global_freq_df = (
    merged_df.groupby(["unit_index", "unit_id"], as_index=False)
    .agg(
        segment_count=("segment_id", "size"),
        clip_count=("clip_id", "nunique"),
        mean_duration_sec=("duration_sec", "mean"),
        median_duration_sec=("duration_sec", "median"),
        min_duration_sec=("duration_sec", "min"),
        max_duration_sec=("duration_sec", "max"),
    )
)

global_freq_df["segment_proportion"] = global_freq_df["segment_count"] / max(n_total_segments, 1)
global_freq_df["clip_proportion"] = global_freq_df["clip_count"] / max(n_total_clips, 1)
global_freq_df = global_freq_df.sort_values(["segment_count", "unit_index"], ascending=[False, True]).reset_index(drop=True)

if GLOBAL_FREQ_CSV.exists() and not overwrite:
    print(f"global_token_frequencies.csv already exists and overwrite=false -> keeping existing file: {GLOBAL_FREQ_CSV}")
    logger.info("global_token_frequencies.csv exists and overwrite=false; file not rewritten")
else:
    global_freq_df.to_csv(GLOBAL_FREQ_CSV, index=False)
    print("Saved:", GLOBAL_FREQ_CSV)
    logger.info("Saved global_token_frequencies.csv to %s", GLOBAL_FREQ_CSV)

display(global_freq_df.head(20))
print("Total unique units in global table:", len(global_freq_df))
print("Top 10 most frequent units:", global_freq_df["unit_id"].head(10).tolist())

In [ ]:
# --- Final sanity summary ---
clip_seq_lengths = pd.Series([rec["n_segments"] for rec in sequence_records], name="n_segments")
display(clip_seq_lengths.describe())

print("Any clips missing a sequence?", bool(len(sequence_records) == 0))
print("Any segment rows missing unit_id?", bool(merged_df["unit_id"].isna().any()))
print("Any duplicated segment_id after assignment?", bool(merged_df["segment_id"].duplicated().any()))
print("Selected K from model bundle:", selected_k)
print("Unique units actually used:", merged_df["unit_index"].nunique())
print("Log file:", LOG_FILE)

logger.info("04_assign_unit_tokens completed successfully")